In [18]:
import sys
sys.path.append("..")
from src.features import *
from src.data.make_dataset import *
from src.data.weights import *
from src.features.zoning_nonconformity_scripts import *
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
test_merge("Newton", "base")

{'Single Family', 'No Residential Uses Allowed', 'More than Eight Units', 'Two Family'}
Split Zoned Parcels Detected
Clean Join successful?
True
Any Parcels missing a Zone Code?
2
Same number of parcels we started with?
True
Parcels with Zoning Table Joined
1410
Zones assigned to Parcels by Largest Share
{'Two Family', 'More than Eight Units', nan, 'No Residential Uses Allowed', 'Single Family'}


In [ ]:
zoning_layer

In [3]:
res_zoning = get_zoning_data("Melrose", type = "overlay")

importing overlay
     oid_  zo_code  muni_id     muni zo_abbr zo_name  shape_leng  \
215   NaN   178SGD    178.0  Melrose     SGD    None         NaN   
216   NaN  178RCOD    178.0  Melrose    RCOD    None         NaN   
217   NaN  178RCOD    178.0  Melrose    RCOD    None         NaN   

     Shape_Length    Shape_Area  \
215   1407.703217  76041.543643   
216   1911.791208  68915.708418   
217   1005.224495  31781.615530   

                                              geometry  
215  MULTIPOLYGON (((235335.578 910043.445, 235107....  
216  MULTIPOLYGON (((235424.051 912201.915, 235410....  
217  MULTIPOLYGON (((235490.263 912112.999, 235488....  
    ZO_CODE                         ZO_NAME  MUNI_ID     MUNI  \
41      178             Floodplain District      178  Melrose   
42  178RCOD  Rail Corridor Overlay District      178  Melrose   
43   178SGD           Smart Growth District      178  Melrose   

                      ZO_AldUse ZO_ABBR  ZO_USETY  \
41  No Residential Uses Al

In [ ]:
res_zoning

In [25]:
from src.data.make_dataset import zoning_layer    
zoning_project_dir = r'C:\Users\ziacovino\OneDrive - Metropolitan Area Planning Council\Metro Mayors Housing Task Force\Phase 2 Scope of Work\Rightsizing Zoning Project\Data'



# shapefile needs some data updating work
zoning = zoning_layer[zoning_layer['muni'] == "Newton"]
zoning = zoning.to_crs(mass_mainland_crs)
#zo_code = 'Zoning'


# regulation table (exported 4/10)
reg_table_fp = os.path.join(zoning_project_dir, "zoning-atlas-mmc.csv")
#original Newton table
#reg_table_fp = os.path.join(zoning_project_dir, "zoning-regs-by_right.csv")
reg_table = pd.read_csv(reg_table_fp)
reg_table['PCTLOTCOV'] = pd.to_numeric(reg_table['PCTLOTCOV'].str.strip('%'))

In [6]:
print(muni_w_zoning['ZO_AldUse'].isna())

0        False
1        False
2        False
3        False
4        False
         ...  
23942    False
23943    False
23944    False
23945    False
23946    False
Name: ZO_AldUse, Length: 23947, dtype: bool


In [2]:
par_gdf = get_landuse_data("Newton")
zon_gdf = get_zoning_data("Newton")
muni_w_zoning= zoning_merge(zoning_gdf= zon_gdf, parcels_gdf= par_gdf)
bld_footprint = ldr_bld[ldr_bld['CITY'] == "Newton"]

    ZO_CODE                ZO_NAME  MUNI_ID    MUNI  \
201  207BU1             Business 1      207  Newton   
202  207BU2             Business 2      207  Newton   
203  207BU4             Business 4      207  Newton   
204  207BU5             Business 5      207  Newton   
205  207LMD  Limited manufacturing      207  Newton   

                       ZO_AldUse ZO_ABBR  ZO_USETY  \
201        More than Eight Units     BU1       2.0   
202        More than Eight Units     BU2       2.0   
203        More than Eight Units     BU4       2.0   
204        More than Eight Units     BU5       2.0   
205  No Residential Uses Allowed     LMD       2.0   

                                              ZO_USEDE  MINLOTSIZE PCTLOTCOV  \
201  Office, bank. Retail, theater, restaurant with...     10000.0       NaN   
202  Uses alloed in B 1, wholesale. Bowling alley, ...     10000.0       NaN   
203                                Uses allowed in B 1     40000.0       50%   
204                     

In [ ]:
bld_footprint['ftpt_area'] = bld_footprint.area

bld_merge_test = structure_merge(parcels_gdf=par_gdf, roofprints_gdf= bld_footprint)

In [4]:
bld_merge_test['floors'] = round(bld_merge_test['MEDIAN_stories']*4)/4 #rounds to quarter story baed on median height
bld_merge_test['height'] = bld_merge_test['MEDIAN']*3.8084
#moving the filter around
bld_merge_test = bld_merge_test[bld_merge_test[units]> 2]

analysis_test = zoning_merge(zoning_gdf= zon_gdf, parcels_gdf= bld_merge_test)




Parcel Rows
2881
Unique LOC_IDS
2881
Split Zoned Parcels Detected
Clean Join successful?
True
Any Parcels missing a Zone Code?
149
Same number of parcels we started with?
True
Parcels with Zoning Table Joined
2881
Zones assigned to Parcels by Largest Share


In [ ]:
analysis_test

In [9]:
print(len(set(par_gdf['LOC_ID'])))
print(len(set(muni_w_zoning['LOC_ID'])))
print(len(bld_footprint['LOC_ID_bld']))
print(len(set(bld_footprint['LOC_ID_bld'])))

muni_parcels = muni_w_zoning.merge(bld_footprint, left_on = 'LOC_ID', right_on = 'LOC_ID_bld', how = "outer")
len(set(muni_parcels['LOC_ID']))

23947
23772
22872
22436


23773

In [ ]:
muni_parcels

In [ ]:
arcpy.ImportToolbox(r"\\Data-Sync\Public\DataServices\Projects\Current_Projects\Environment\PDM\project_files\.ArcPro\01_PDM_Initialization_Edits\PDM_Initialization_Edits_TableTool.atbx")
arcpy.PDMInitializationEditsTableToolatbx.ExportExistingFeatures1(
    TownName="ASHLAND",
    Project_Folder=r"\\Data-Sync\Public\DataServices\Projects\Current_Projects\Environment\PDM\project_files\Ashland\2024_Update\Updated_Features",
    Layout="PDM_UpdatedFeatures"
)

In [ ]:
par_gdf

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
        ..
2233   NaN
2234   NaN
2235   NaN
2236   NaN
2237   NaN
Name: MAX_GFA, Length: 2238, dtype: float64


In [5]:
zoning_table = pd.DataFrame(zon_gdf.drop(columns= ['geometry', 'Shape_Length', 'Shape_Area', 'EDITDATE', 'CREATEDDATE']))
zoning_table = zoning_table.drop_duplicates()

In [2]:
zoning_table['floors'] = -99

NameError: name 'zoning_table' is not defined

In [4]:
parcels_gdf = par_gdf
print("Rows: ", len(parcels_gdf['LOC_ID'])) 
print("Unique LOCIDS: ", len(set(parcels_gdf['LOC_ID'])))
zoning_gdf = zon_gdf

par_zon_join = parcels_gdf.overlay(zoning_gdf, how = "intersection", keep_geom_type = True)
print("Overlay")
print(len(par_zon_join['LOC_ID']) >  len(set(par_zon_join['LOC_ID'])))
print("Rows: ", len(par_zon_join['LOC_ID'])) 
print("Unique LOCIDS: ", len(set(par_zon_join['LOC_ID'])))

par_zon_sjoin= parcels_gdf.sjoin(zoning_gdf)
print("sjoin")
print(len(par_zon_sjoin['LOC_ID']) >  len(set(par_zon_sjoin['LOC_ID'])))
print("Rows: ", len(par_zon_sjoin['LOC_ID'])) 
print("Unique LOCIDS: ", len(set(par_zon_sjoin['LOC_ID'])))



Rows:  23947
Unique LOCIDS:  23947
Overlay
True
Rows:  24068
Unique LOCIDS:  23772
sjoin
True
Rows:  29787
Unique LOCIDS:  23863


In [5]:

print ("Split Zoned Parcels Detected")       
# determine the proportion of the total parcel area
par_zon_join['zone_area'] = par_zon_join['geometry'].area
par_zon_join['zone_share'] = par_zon_join.apply(lambda row: row['zone_area']*10.7639/row['LOT_SIZE_GIS'], axis=1) #its a row

# ID the index of the row with the largest share of a parcel in a zone for each unique LOCID, reset_index() makes into a df
idx = par_zon_join.fillna(999999).groupby('LOC_ID')['zone_share'].idxmax().reset_index()

# subset the original spatial join to the rows where the share is the largest for each unique LOC ID--should be one row for each LOCID again
clean_join = par_zon_join.loc[idx['zone_share']]
print("Clean Join successful?")
print(len(idx) == len(clean_join))
print("Index: ", len(idx))
print("Clean Join", len(clean_join))
# cut that table to just the LOC ID and the ZO Code, confirm pd
par_zon_xwalk = clean_join[['LOC_ID','ZO_CODE']]
# join the zone code onto the parcels, double check the join
parcels_zone_rec = pd.merge(parcels_gdf, par_zon_xwalk, left_on= 'LOC_ID', right_on= "LOC_ID", how = "left")
print("Zone Code succesfully joined?")
print(len(parcels_zone_rec['LOC_ID']) == len(clean_join))

print("Zoning Reconciled Parcel:", len(parcels_zone_rec))

# take the zoning input and get rid of the geometry so we can do non-spatial joins
zoning_table = pd.DataFrame(zoning_gdf.drop(columns= 'geometry'))
# join the rest of the zoning table back to the parcels
par_zon_join_fixed = pd.merge(parcels_zone_rec, zoning_table, left_on= 'ZO_CODE', right_on= 'ZO_CODE', how = "inner")
print("Zones assigned to Parcels by Largest Share")


Split Zoned Parcels Detected


Clean Join successful?
True
Index:  23772
Clean Join 23772
Zone Code succesfully joined?
False
Zoning Reconciled Parcel: 23947
Zones assigned to Parcels by Largest Share


In [ ]:
# merge_test['zo_code'].unique()
len(zon_gdf)
print("Rows: ", len(merge_test['LOC_ID'])) 
print("Unique LOCIDS: ", len(set(merge_test['LOC_ID'])))


Unique LOCIDS:  23772


Lot Coverage De-Bugging

In [3]:
parcel_size_criteria = calculate_overlap(layer_1= muni_w_zoning,
                                        layer_2 = bld_footprint,
                                        how = "percent",
                                        new_field_name = 'par_lot_cov')

In [23]:
ldr_bld['ftpt_area'] = ldr_bld.area
ldr_bld_join = ldr_bld[['LOC_ID_bld', 'PCT75', 'PCT75_stories', 'ftpt_area']]
par_gdf = par_gdf.merge(ldr_bld_join, left_on = 'LOC_ID', right_on = 'LOC_ID_bld')


In [24]:
par_gdf['floors'] = round(par_gdf['PCT75_stories']*4)/4
par_gdf['height'] = par_gdf['PCT75']*3.8084


In [26]:
par_gdf['GFA']= par_gdf.apply(lambda row: (row['ftpt_area']*10.7639)*row['PCT75_stories'], axis = 1)

In [30]:
par_gdf['gfa_comp'] = round(par_gdf['GFA'] - par_gdf['BLD_AREA'])

par_gdf['gfa_comp']

0        -434.0
1        -906.0
2       -2109.0
3        -235.0
4       -1351.0
          ...  
22867     255.0
22868    1900.0
22869    -773.0
22870    -317.0
22871     394.0
Name: gfa_comp, Length: 22872, dtype: float64

In [4]:
#def calculate_overlap (
layer_1 = muni_w_zoning
layer_2 = bld_footprint_dis
how = 'percent'
new_field_name = 'par_lot_cov'
normalize = False
inverse =False
buffer =None


In [6]:
os.path.join(zoning_project_dir, "zoning-atlas-mmc.csv")

NameError: name 'zoning_project_dir' is not defined

In [ ]:

# '''
# For a given polygon base layer, calculates either 
# 1) the total area of overlap (in meters squared) with a polygon layer of interest;
# 2) the percentage of the base layer that is overlapped by a polygon layer of interest; OR 
# 3) the total length of a line layer of interest that is contained by the base layer [need to do this]


# INPUTS: 
# - layer_1: (GeoDataFrame, polygon) The base layer being enriched 
# - layer_2: (GeoDataFrame, polygon or line) The overlap layer of interest 
# - how: (string, default 'area') the type of calculation: 
#     - 'area' = calculates the area of overlap in meters squared 
#     - 'percent' = calculate the percentage of the base layer  
#     - 'length' = calculate the length (in m) of an overlapping line layer within a chosen distance from layer_1
# - new_field_name: (string)  Input a string to represent this layer in the output dataset 

# OPTIONAL PARAMETER(S):
# - normalize: (bool, default True) When True, adds an additional field containing a normalized value (0-1 scale) of the area/percentage of overlap. 
# - inverse: (bool, default False) When True, normalized values are scored in the inverse. This implies that greater overlap = less suitability.  
# - buffer: (int, default None) For calculating length, specify a buffer distance around layer_1 to search

# OUTPUT:
# Fields added to layer_1: 
# - Overlap values:  
#     - '[new_field_name]_sqm' or '[new_field_name]_pct' or '[new_field_name]_m'
# - Normalized overlap values (with inverse applied if selected):  
#     - '[new_field_name]_sqm_n' or '[new_field_name]_pct_n' or '[new_field_name]_m_n'

# '''
from src.data.make_dataset import id_field
# if not how:
# how = 'area'

# valid = {'area', 'percent', 'length'}
# if how not in valid:
# raise ValueError("how must be one of %r." % valid)

#reproject all to mass mainland
mass_mainland_crs = "EPSG:26986"
layer_1 = layer_1.to_crs(mass_mainland_crs)
layer_2 = layer_2.to_crs(mass_mainland_crs)
print("Transformation Sucessful")

#make a list of original columns for later
layer_1_fields = layer_1.columns.tolist()
print("Listed Columns")
#if how in ['area' , 'percent']:

#only keep parts of layer 1 that intersects with layer 2
#ZI Change: keep geom = True
intersection_layer = layer_1.overlay(layer_2, how='intersection', keep_geom_type=True)


In [3]:
bld_footprint['bld_area_sqm']= bld_footprint['geometry'].area
bld_footprint = pd.DataFrame(bld_footprint.drop(columns = 'geometry'))
bld_footprint = bld_footprint[['LOC_ID_bld', 'bld_area_sqm']]

c:\Users\ziacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [13]:
muni_w_zoning= zoning_merge(zoning_gdf= zon_gdf, parcels_gdf= par_gdf)

Split Zoned Parcels Detected
Clean Join successful?
True
Zone Code succesfully joined?
2
True
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 23947 entries, 0 to 23946
Data columns (total 64 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   LOC_ID              23947 non-null  object  
 1   geometry            23947 non-null  geometry
 2   Unnamed: 0          23947 non-null  int64   
 3   TOWN_ID             23947 non-null  int64   
 4   PROP_ID             23947 non-null  object  
 5   BLDG_VAL            23947 non-null  int64   
 6   LAND_VAL            23947 non-null  int64   
 7   OTHER_VAL           23947 non-null  int64   
 8   TOTAL_VAL           23947 non-null  int64   
 9   FY                  23947 non-null  object  
 10  LOT_SIZE            23784 non-null  float64 
 11  LS_DATE             23947 non-null  object  
 12  LS_PRICE_S          23947 non-null  float64 
 13  LS_PRICE_L          23947 non-null

In [4]:
muni_w_zoning = muni_w_zoning.merge(bld_footprint, left_on= "LOC_ID", right_on = "LOC_ID_bld")

In [26]:
print(len(set(bld_footprint['LOC_ID_bld'])))
print(len(set(muni_w_zoning['LOC_ID'])))
print(len(muni_w_zoning))
print(len(par_gdf))
print(len(zon_gdf))

mismatch = bld_footprint['LOC_ID_bld'].isin(muni_w_zoning['LOC_ID'])
print(mismatch)

22303
23772
9951458
23947
2238
227934    True
227935    True
227936    True
227937    True
227938    True
          ... 
574685    True
574686    True
574687    True
574688    True
574689    True
Name: LOC_ID_bld, Length: 57976, dtype: bool


In [ ]:

#get area of overlap for the area of intersection
intersection_layer[new_field_name + '_sqm'] = intersection_layer['geometry'].area  
intersection_layer = intersection_layer.groupby(by=id_field).agg({(new_field_name + '_sqm'):'sum'}).reset_index()


#join back to parcels data, remove additional rows with groupby
layer_1_enriched = layer_1.merge(intersection_layer[[id_field, (new_field_name + '_sqm')]], 
                                on=id_field, 
                                how='left')

#try a fillna() to account for np.nan in overlap values
layer_1_enriched[new_field_name + '_sqm'] = layer_1_enriched[new_field_name + '_sqm'].fillna(0)

#get percent of overlap for each feature in the base layer
layer_1_enriched[new_field_name + '_pct'] = (layer_1_enriched[new_field_name + '_sqm'] / (layer_1_enriched['geometry'].area)) 

#final output defined by input and optional parameters
# if how == 'area':
#     if normalize: 
#         layer_1_enriched[(new_field_name + '_sqm_n')] = normalize_field(layer_1_enriched, (new_field_name + '_sqm'))
#         if inverse:
#             layer_1_enriched[(new_field_name + '_sqm_n')] = 1 - layer_1_enriched[(new_field_name + '_sqm_n')] 
#         layer_1_enriched = layer_1_enriched[layer_1_fields + [(new_field_name + '_sqm'), (new_field_name + '_sqm_n')]]
#     else:
#         layer_1_enriched = layer_1_enriched[layer_1_fields + [(new_field_name + '_sqm')]]

# elif how == 'percent':
#     if normalize: 
#         layer_1_enriched[(new_field_name + '_pct_n')] = normalize_field(layer_1_enriched, [(new_field_name + '_pct')])
#         if inverse:
#             layer_1_enriched[(new_field_name + '_pct_n')] = 1 - layer_1_enriched[(new_field_name + '_pct_n')] 
#         layer_1_enriched = layer_1_enriched[layer_1_fields + [(new_field_name + '_pct'), (new_field_name + '_pct_n')]]
#     else:
layer_1_enriched = layer_1_enriched[layer_1_fields + [(new_field_name + '_pct')]]

# else: #for line length overlaps
# layer_1_buffer = buffer_gdf(layer_1, buffer)

# #Intersect buffered parcels with lines - returns line segments that intersect with each parcel
# intersection_layer = gpd.overlay(df1=layer_1_buffer, 
#                                 df2=layer_2, 
#                                 how="intersection", 
#                                 keep_geom_type=False)

# #sum line length per unique ID
# intersection_layer[new_field_name + '_m'] = intersection_layer['geometry'].length 
# intersection_layer[new_field_name + '_m'] = intersection_layer.groupby(id_field)[new_field_name + '_m'].transform("sum")

# #merge length field back to layer_1
# layer_1_enriched = layer_1.merge(intersection_layer[[id_field, (new_field_name + '_m')]], 
#                                     on=id_field,
#                                     how='left').fillna(0).drop_duplicates()

# if normalize: 
#     layer_1_enriched[(new_field_name + '_m_n')] = normalize_field(layer_1_enriched, (new_field_name + '_m'))
#     if inverse:
#         layer_1_enriched[(new_field_name + '_m_n')] = 1 - layer_1_enriched[(new_field_name + '_m_n')] 
#     layer_1_enriched = layer_1_enriched[layer_1_fields + [(new_field_name + '_m'), (new_field_name + '_m_n')]]
# else:
#     layer_1_enriched = layer_1_enriched[layer_1_fields + [(new_field_name + '_m')]]

# return layer_1_enriched

In [ ]:
parcel_size_criteria

In [ ]:
# return 1 if there is more building than the regulated percent lot coverage allows
def label_lotcov (row):
    if row['par_lot_cov_pct'] > row['PCTLOTCOV']:
        return 1
    else: 
        return 0

        
parcel_size_criteria['lc_conf'] = parcel_size_criteria.apply(lambda row:
                                                            label_lotcov(row),
                                                            axis = 1)